[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_10/MFM_ch10_intraday_momentum/MFM_ch10_intraday_momentum.ipynb)

# MFM_ch10_intraday_momentum

Case study, Baltussen, Da, Lammers and Martens (2021, JFE): market intraday momentum. The return of the last half hour before the close (15:30-16:00) is regressed on the return of the rest of the day (previous close to 15:30, dividend added on ex-dividend dates), their equation (7), with Newey-West standard errors (5 lags); SPY 5-minute bars of the regular session, days with all 78 bars and a complete previous day, October 2020 - September 2026; slope by year, on down and up days, for last intervals of 10 to 60 minutes, the Gao, Han, Li and Zhou (2018) predictor (previous close to 10:00), the timing strategy and the next-day reversal; the paper's pooled equity futures slope (100 beta = 4.18) as reference.

*Modelling Financial Markets (MFM), Chapter 10: Market Microstructure. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_10'
    os.makedirs(OUT_DIR, exist_ok=True)
    os.chdir(OUT_DIR)  # relative chart and CSV outputs persist in Drive
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
GROUP_COL = {'US': MainBlue, 'BVB': IDAred, 'Crypto': Amber}

HERE = '.'
SEED = 42
START2 = '2024-09-19'
START10 = '2016-09-19'
B_BOOT = 1000


def save_fig(name):
    """Save the figure as a transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles, labels, ncol=3, y=0.0):
    """A single legend for a multi-panel figure, below the figure."""
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# use the local copy of the course data when the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# key -> (symbol, label, group)
ASSETS = {
    'SPY': ('SPY.US', 'SPY', 'US'), 'AAPL': ('AAPL.US', 'Apple', 'US'), 'MSFT': ('MSFT.US', 'Microsoft', 'US'),
    'NVDA': ('NVDA.US', 'NVIDIA', 'US'), 'JPM': ('JPM.US', 'JPMorgan', 'US'), 'GME': ('GME.US', 'GameStop', 'US'),
    'MSTR': ('MSTR.US', 'Strategy', 'US'),
    'TLV': ('TLV.RO', 'Banca Transilvania', 'BVB'), 'SNP': ('SNP.RO', 'OMV Petrom', 'BVB'),
    'H2O': ('H2O.RO', 'Hidroelectrica', 'BVB'), 'BRD': ('BRD.RO', 'BRD', 'BVB'), 'SNG': ('SNG.RO', 'Romgaz', 'BVB'),
    'SNN': ('SNN.RO', 'Nuclearelectrica', 'BVB'), 'TGN': ('TGN.RO', 'Transgaz', 'BVB'),
    'EL': ('EL.RO', 'Electrica', 'BVB'), 'DIGI': ('DIGI.RO', 'Digi', 'BVB'), 'TEL': ('TEL.RO', 'Transelectrica', 'BVB'),
    'TVBETETF': ('TVBETETF.RO', 'BET ETF', 'BVB'),
    'BTC': ('BTC-USD.CC', 'Bitcoin', 'Crypto'), 'ETH': ('ETH-USD.CC', 'Ethereum', 'Crypto'),
    'SOL': ('SOL-USD.CC', 'Solana', 'Crypto'), 'XRP': ('XRP-USD.CC', 'XRP', 'Crypto'),
    'DOGE': ('DOGE-USD.CC', 'Dogecoin', 'Crypto'), 'ADA': ('ADA-USD.CC', 'Cardano', 'Crypto'),
    'LTC': ('LTC-USD.CC', 'Litecoin', 'Crypto'),
}
GROUPS = {g: [k for k, v in ASSETS.items() if v[2] == g] for g in ('US', 'BVB', 'Crypto')}
LABELS = {k: v[1] for k, v in ASSETS.items()}
# large adjustment events that are cash dividends, not splits (issuer announcement):
#   SNN.RO 2018-12-21: gross dividend of 1.61 lei per share, ex-date 21 December 2018
#   (https://www.nuclearelectrica.ro/wp-content/uploads/2018/12/SNN_Comunicat-plata-dividende-suplimentare_EN.pdf)
CASH_DIVIDENDS = {'SNN.RO': ['2018-12-21']}

_CACHE = {}


def read_market(symbol):
    """Read the daily price file of one symbol, from a local copy or from the GitHub repository."""
    if symbol in _CACHE:
        return _CACHE[symbol]
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    _CACHE[symbol] = pd.read_csv(src, index_col='date', parse_dates=True).sort_index()
    return _CACHE[symbol]


def read_reference_rate(currency='USD', start='2014-01-01', end=END):
    """Official RON reference rate (BNR yearly XML archives)."""
    key = ('REF', currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'rate']).drop_duplicates('date').set_index('date')['rate']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def _clean(key, d):
    d = d[(d['high'] > d['low']) & (d['volume'].fillna(0) > 0)]
    if ASSETS[key][2] != 'Crypto':
        d = d[d.index.dayofweek < 5]
        # isolated bad ticks: adjusted price more than twice / less than half the local median (21 days)
        med = d['adjusted_close'].rolling(21, center=True, min_periods=5).median()
        d = d[np.abs(np.log(d['adjusted_close'] / med)) < np.log(2)]
    return d


def ohlc(key, start='2016-09-19', end=END):
    """Open, high, low, close scaled by the adjustment factor of the adjusted price (dividends and splits)."""
    d = read_market(ASSETS[key][0]).loc[start:end].copy()
    f = d['adjusted_close'] / d['close']
    for c in ['open', 'high', 'low', 'close']:
        d[c] = d[c] * f
    return _clean(key, d)[['open', 'high', 'low', 'close', 'volume']]


def split_adjusted_close(d, symbol=None):
    """Close price adjusted ONLY for splits and bonus shares (same basis as the volume in number of shares);
    the large cash dividends in CASH_DIVIDENDS are not treated as splits."""
    k = (d['close'].shift(1) / d['close']) / (d['adjusted_close'].shift(1) / d['adjusted_close'])
    k = k.where(np.abs(np.log(k)) > 0.15, 1.0).fillna(1.0)      # split days: the split ratio
    for day in CASH_DIVIDENDS.get(symbol, []):
        k.loc[k.index == pd.Timestamp(day)] = 1.0
    back = k[::-1].cumprod()[::-1].shift(-1).fillna(1.0)          # product of the splits after each day
    return d['close'] / back


def returns(key, start='2016-09-19', end=END):
    """Daily log returns from the adjusted price (days without trading dropped)."""
    d = read_market(ASSETS[key][0]).loc[start:end]
    d = _clean(key, d)
    return np.log(d['adjusted_close']).diff().dropna().rename(key)


def dollar_volume(key, start='2016-09-19', end=END):
    """Daily traded value, in USD."""
    d = read_market(ASSETS[key][0]).loc[:end]
    grp = ASSETS[key][2]
    if grp == 'Crypto':
        dv = d['volume']                                          # already in USD
    else:
        dv = split_adjusted_close(d, ASSETS[key][0]) * d['volume']
        if grp == 'BVB':                                          # RON -> USD at the BNR reference rate
            fx = read_reference_rate('USD', start='2014-01-01', end=end)
            dv = dv / fx.reindex(dv.index).ffill()
    d = _clean(key, d.assign(dv=dv))
    return d['dv'].loc[start:end].dropna().rename(key)


def intraday_spy():
    """SPY 5-minute bars, regular session (09:30-15:55 = bar start), complete days only (78 bars)."""
    fname = 'intraday/SPY.US_5m.csv'
    path = os.path.join(MARKET_DIR, fname)
    d = pd.read_csv(path if os.path.exists(path) else REPO_RAW + fname, parse_dates=['datetime_utc'])
    t = d['datetime_utc'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)
    d = d.assign(t=t, date=t.dt.normalize(), tod=t.dt.strftime('%H:%M')).drop(columns='datetime_utc')
    d = d[(d['tod'] >= '09:30') & (d['tod'] <= '15:55')]
    n = d.groupby('date').size()
    d = d[d['date'].isin(n[n == 78].index)].set_index('t').sort_index()
    d['r'] = np.log(d['close']).groupby(d['date']).diff()
    first = d['tod'] == '09:30'
    d.loc[first, 'r'] = np.log(d.loc[first, 'close'] / d.loc[first, 'open'])   # first bar: from the open
    d['dv'] = d['close'] * d['volume']
    return d


def intraday_btc():
    """Bitcoin 5-minute bars (UTC), 24 hours a day; volume is not used (missing on many bars)."""
    fname = 'intraday/BTC-USD.CC_5m.csv'
    path = os.path.join(MARKET_DIR, fname)
    d = pd.read_csv(path if os.path.exists(path) else REPO_RAW + fname, parse_dates=['datetime_utc'])
    d = d.set_index('datetime_utc').sort_index()
    d = d[~d.index.duplicated()]
    d['date'] = d.index.normalize()
    d['r'] = np.log(d['close']).diff()
    gap = d.index.to_series().diff() != pd.Timedelta('5min')
    d.loc[gap, 'r'] = np.nan                                       # no returns across missing bars
    return d

## Helper functions

In [ ]:
# Baltussen, Da, Lammers & Martens (2021): slopes and R2 x 100 (Table 2, Panel A, col. 3; Table 7)
BDLM = dict(beta_eq=4.18, t_eq=7.29, r2_eq=2.45, r2oos_eq=2.88, beta_onfh_eq=4.86, beta_nge_neg=6.63, t_nge_neg=4.78,
            beta_nge_pos=0.82, t_nge_pos=1.03, rev_t3=-27.98)
LAGS = 5
CUTS = ['14:55', '15:05', '15:15', '15:25', '15:35', '15:45']
CUT0 = '15:25'

## Analysis

In [ ]:
def momentum_days():
    """One row per day: r_ROD, r_LH, r_ONFH, log prices at the cut points (relative to the previous close), next-day return."""
    s = intraday_spy()
    c = s.pivot_table(index='date', columns='tod', values='close', dropna=False)
    d = read_market('SPY.US').copy()
    d.index = pd.to_datetime(d.index)
    prev_cal = pd.Series(d.index[:-1], index=d.index[1:])
    f = d['adjusted_close'] / d['close']
    div = np.log(f / f.shift(1)).reindex(c.index).fillna(0.0)          # dividend yield on ex-dividend days
    prev_day = pd.Series(c.index, index=c.index).shift(1)
    ok = prev_day.eq(prev_cal.reindex(c.index))                       # previous trading day also complete
    pc = c['15:55'].shift(1)
    lp = np.log(c.div(pc, axis=0)).add(div, axis=0)                   # log price relative to the previous close
    out = pd.DataFrame({'rod': lp[CUT0], 'lh': lp['15:55'] - lp[CUT0], 'onfh': lp['09:55'], 'close': lp['15:55']})
    for k in CUTS:
        out['x' + k.replace(':', '')] = lp[k]
    out['next'] = np.log(c['15:55'].shift(-1) / c['15:55']) + div.shift(-1)   # close to next close
    nxt_ok = ok.shift(-1, fill_value=False)
    out.loc[~nxt_ok, 'next'] = np.nan
    out = out[ok.values].dropna(subset=['rod', 'lh', 'onfh', 'close'] + ['x' + k.replace(':', '') for k in CUTS])
    out['year'] = out.index.year
    return out


def nw(y, x):
    """OLS of y on a constant and x, Newey-West (LAGS) t; slope and R2 x 100, 95% interval of 100 beta."""
    m = sm.OLS(np.asarray(y, float), sm.add_constant(np.asarray(x, float))).fit(cov_type='HAC', cov_kwds={'maxlags': LAGS})
    b, se = 100 * m.params[1], 100 * m.bse[1]
    return dict(n=int(len(y)), beta=float(b), se=float(se), t=float(m.tvalues[1]), lo=float(b - 1.96 * se),
                hi=float(b + 1.96 * se), r2=float(100 * m.rsquared))


def analysis(D):
    res = dict(paper=BDLM, n=int(len(D)), first=str(D.index[0].date()), last=str(D.index[-1].date()))
    res['rod'] = nw(D['lh'], D['rod'])
    res['onfh'] = nw(D['lh'], D['onfh'])
    res['down'] = nw(D.loc[D['rod'] < 0, 'lh'], D.loc[D['rod'] < 0, 'rod'])
    res['up'] = nw(D.loc[D['rod'] >= 0, 'lh'], D.loc[D['rod'] >= 0, 'rod'])
    res['years'] = {}
    yb = D['year'].clip(lower=2021)                                     # Oct-Dec 2020 pooled with 2021
    for y, g in D.groupby(yb):
        res['years']['2020-21' if y == 2021 else str(y)] = nw(g['lh'], g['rod'])
    eta = np.where(D['rod'] > 0, D['lh'], -D['lh'])
    res['eta'] = dict(mean_bp=float(1e4 * eta.mean()), ann_pct=float(100 * 252 * eta.mean()),
                      sr=float(eta.mean() / eta.std(ddof=1) * np.sqrt(252)), hit=float((eta > 0).mean()))
    k = D['next'].notna()
    res['reversal'] = nw(D.loc[k, 'next'], D.loc[k, 'lh'])
    res['cuts'] = {}
    for c in CUTS:
        x = D['x' + c.replace(':', '')]
        res['cuts'][c] = nw(D['close'] - x, x)
    q = pd.qcut(D['rod'], 10, labels=False)
    g = D.groupby(q)
    res['bins'] = dict(x=list(100 * g['rod'].mean()), y=list(1e4 * g['lh'].mean()),
                       se=list(1e4 * g['lh'].std(ddof=1) / np.sqrt(g.size())))
    res['sd_rod'] = float(100 * D['rod'].std(ddof=1))
    res['sd_lh_bp'] = float(1e4 * D['lh'].std(ddof=1))
    res['mean_rod'] = float(100 * D['rod'].mean())
    res['mean_lh_bp'] = float(1e4 * D['lh'].mean())
    res['alpha_bp'] = float(1e4 * (D['lh'].mean() - res['rod']['beta'] / 100 * D['rod'].mean()))
    return res


def fig_intraday_momentum(R):
    from matplotlib.lines import Line2D
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4), gridspec_kw=dict(width_ratios=[1.15, 1]))
    B = R['bins']
    x, y, se = np.array(B['x']), np.array(B['y']), np.array(B['se'])
    a1.errorbar(x, y, yerr=1.96 * se, fmt='o', color=MainBlue, ms=4, lw=0.9, capsize=2)
    xs = np.linspace(x.min() - 0.3, x.max() + 0.3, 50)
    a1.plot(xs, R['alpha_bp'] + R['rod']['beta'] * xs, color=MainBlue, lw=1.2)
    a1.plot(xs, R['mean_lh_bp'] + BDLM['beta_eq'] * (xs - R['mean_rod']), color=IDAred, lw=1.2, ls='--')
    a1.axhline(0, color=Navy, lw=0.6)
    a1.set_xlabel('Rest-of-day return, previous close to 15:30 (%)')
    a1.set_ylabel('Last-half-hour return,\n15:30 to 16:00 (bp)')
    yr = R['years']
    ks = list(yr)
    xx = np.arange(len(ks))
    b = np.array([yr[k]['beta'] for k in ks])
    a2.errorbar(xx, b, yerr=[b - np.array([yr[k]['lo'] for k in ks]), np.array([yr[k]['hi'] for k in ks]) - b],
                fmt='s', color=MainBlue, ms=4, lw=0.9, capsize=2)
    a2.axhline(BDLM['beta_eq'], color=IDAred, lw=1.2, ls='--')
    a2.axhline(0, color=Navy, lw=0.6)
    a2.set_xticks(xx)
    a2.set_xticklabels(ks)
    a2.set_ylabel('Slope 100 beta')
    r = R['rod']
    h = [Line2D([], [], color=MainBlue, marker='o', ls='none', ms=4),
         Line2D([], [], color=MainBlue, lw=1.2),
         Line2D([], [], color=MainBlue, marker='s', ls='none', ms=4),
         Line2D([], [], color=IDAred, lw=1.2, ls='--')]
    lab = ['SPY decile means of the last-half-hour return, 95% interval',
           f"SPY OLS fit: 100 beta = {r['beta']:.2f} (95% interval {r['lo']:.2f} to {r['hi']:.2f})",
           'SPY slope by year, Newey-West 95% interval',
           'Paper, pooled equity futures 1974-2020: 100 beta = 4.18']
    fig.legend(h, lab, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=2, frameon=False)
    plt.tight_layout()
    save_fig('ch10_intraday_momentum')

## Run

In [ ]:
D = momentum_days()
R = analysis(D)
fig_intraday_momentum(R)
print({k: R[k] for k in ('n', 'first', 'last', 'rod', 'onfh', 'down', 'up', 'eta', 'reversal')})
pd.DataFrame(R['years']).T.round(3)

In [ ]:
pd.DataFrame(R['cuts']).T.round(3)   # start of the 5-minute bar that opens the last interval

![ch10_intraday_momentum](./ch10_intraday_momentum.png)

Output: `ch10_intraday_momentum.pdf`